In [1]:
import torch
import numpy as np
import pandas as pd
import ast
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

c:\Users\arsal\Desktop\support-ticket-triage-system\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


### Data Formatting
We need to binarize your labels (just like before), but Hugging Face models require these labels to be float32 (decimals) rather than integers, because the neural network outputs probabilities.

In [2]:
df = pd.read_csv("../dataset/dataset_13_labels.csv")
df["text"] = df["title"].fillna("") + " [SEP] " + df["body"].fillna("")
df.drop(["title", "body"], axis=1, inplace=True)

# 1. Ensure labels are actual lists (fixing the string issue from before)
df['labels-mapped'] = df['labels-mapped'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# 2. Binarize the labels
mlb = MultiLabelBinarizer()
binary_labels = mlb.fit_transform(df['labels-mapped'])
num_labels = len(mlb.classes_)

# 3. Add the binarized labels back to the dataframe as a list of floats
df['binary_labels'] = binary_labels.astype(np.float32).tolist()

# 4. Split the data (80% train, 20% test)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# 5. Convert Pandas DataFrames into Hugging Face Datasets
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df[['text', 'binary_labels']]),
    "test": Dataset.from_pandas(test_df[['text', 'binary_labels']])
})

print("Classes:", mlb.classes_)

Classes: ['bug' 'build_ci_cd' 'compatibility' 'documentation' 'feature_request'
 'integration_api' 'maintenance_data' 'performance' 'question'
 'security_access' 'setup_config' 'testing' 'ui_ux']


### Tokenization
This replaces your TfidfVectorizer. The Tokenizer converts your text into a sequence of numbers that DistilBERT understands.

In [3]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # Padding and truncation ensure all sentences are the same length
    tokenized = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)
    # Hugging Face expects the target variable to be named 'labels'
    tokenized["labels"] = examples["binary_labels"]
    return tokenized

# Apply tokenization to the whole dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=['text', 'binary_labels', '__index_level_0__'])

Map: 100%|██████████| 21382/21382 [00:06<00:00, 3100.14 examples/s]


### The Metric Function
Transformers output raw scores (logits). We need to convert those scores into probabilities using a Sigmoid function, and then check if the probability is above 50% (0.5) to call it a "Yes" for that label.

In [4]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    
    # Convert logits to probabilities
    probs = torch.sigmoid(torch.Tensor(logits)).numpy()
    
    # Use 0.5 as a threshold for predicting a label
    predictions = (probs >= 0.5).astype(int)
    
    # Calculate metrics
    macro_f1 = f1_score(labels, predictions, average="macro")
    micro_f1 = f1_score(labels, predictions, average="micro")
    accuracy = accuracy_score(labels, predictions)
    
    return {
        "f1_macro": macro_f1,
        "f1_micro": micro_f1,
        "accuracy": accuracy
    }

### Training Setup & Execution
Here is where we load the actual "Brain" and train it.
Because you have an RTX 4060 (8GB VRAM), we are using a batch size of 16. We are also enabling fp16=True, which allows your RTX card to train the model almost twice as fast using half-precision math.

In [5]:
# 1. Load the pre-trained model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=num_labels, 
    problem_type="multi_label_classification"
)

# 2. Define training rules
training_args = TrainingArguments(
    output_dir="./ticket_triage_model",
    eval_strategy="epoch",    # Check performance at the end of every epoch
    save_strategy="epoch",
    learning_rate=2e-5,             # Standard learning rate for fine-tuning
    per_device_train_batch_size=16, # Fits safely in 8GB VRAM
    per_device_eval_batch_size=16,
    num_train_epochs=3,             # 3 passes over the data is usually plenty
    weight_decay=0.01,
    fp16=True,                      # RTX 4060 Magic! Makes training super fast
    load_best_model_at_end=True,
)

# 3. Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

# 4. START TRAINING!
print("Starting training...")
trainer.train()

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8272.14it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training...


Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,Accuracy
1,0.208037,0.198435,0.559336,0.709696,0.402628
2,0.182808,0.186295,0.628385,0.728906,0.428772
3,0.162955,0.183993,0.650920,0.739424,0.440324


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.61it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=16038, training_loss=0.19681943582320008, metrics={'train_runtime': 1745.2544, 'train_samples_per_second': 147.016, 'train_steps_per_second': 9.189, 'total_flos': 1.6997642570368512e+16, 'train_loss': 0.19681943582320008, 'epoch': 3.0})